# Notebook 04 — Evaluation

Benchmarks the trained LSTM and Transformer models against raw SGP4 propagation on the held-out test set.

**Metrics reported:**
- Mean Absolute Error (MAE, km) and RMSE (km) at T+10, T+30, T+60, T+120 min for each of:
  - SGP4 baseline (no ML)
  - LSTM
  - Transformer

**Key question:** Does the ML model beat SGP4 specifically when the TLE is stale (age > 3 days)?

**Input:** model weights (`lstm_orbit.pt`, `transformer_orbit.pt`), test arrays (`X_test.npy`, `y_test.npy`), `scaler_X.pkl`, `dataset_meta.json`

## 0 — Environment Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running on Google Colab')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'torch', 'sgp4', 'plotly'], check=True)
else:
    print('Running locally')

import io, json, math, pickle, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import torch
import torch.nn as nn
from sgp4.api import Satrec

DATA_DIR = Path('/content') if IN_COLAB else Path('../data/collected')

RE_KM = 6371.0   # mean Earth radius (km) used for error computation in km
print('All imports OK')

## 1 — Load Test Data & Model Weights

**Colab:** upload `dataset.zip` (from Notebook 02) and `lstm_orbit.pt` / `transformer_orbit.pt` (from Notebook 03).  
**Local:** all files are loaded automatically from `data/collected/`.

In [ ]:
if IN_COLAB:
    from google.colab import files as _colab_files
    needed = ['X_test.npy', 'y_test.npy', 'scaler_X.pkl', 'dataset_meta.json',
              'lstm_orbit.pt', 'transformer_orbit.pt']
    missing = [f for f in needed if not (DATA_DIR / f).exists()]
    if missing:
        print(f'Upload: {missing}')
        print('Upload dataset.zip first, then the two .pt weight files.')
        _uploaded = _colab_files.upload()
        for name, data in _uploaded.items():
            if name.endswith('.zip'):
                with zipfile.ZipFile(io.BytesIO(data)) as zf:
                    zf.extractall(DATA_DIR)
                print(f'  Extracted {name}')
            else:
                (DATA_DIR / name).write_bytes(data)
                print(f'  Saved {name}')

X_test = np.load(DATA_DIR / 'X_test.npy')
y_test = np.load(DATA_DIR / 'y_test.npy')   # (N, 4, 3)  lat/lon/alt at 4 horizons

with open(DATA_DIR / 'scaler_X.pkl', 'rb') as f:
    scaler_X = pickle.load(f)
with open(DATA_DIR / 'dataset_meta.json') as f:
    meta = json.load(f)

HORIZONS    = meta['horizons']
FEATURE_COLS = meta['feature_cols']
TARGET_COLS  = meta['target_cols']
N_FEATURES   = X_test.shape[-1]
N_OUT        = y_test.shape[1] * y_test.shape[2]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Test set: {X_test.shape}  Device: {DEVICE}')
print(f'Horizons (min): {HORIZONS}  Targets: {TARGET_COLS}')

In [ ]:
# --- Copy model class definitions needed to load weights ---
# (identical to Notebook 03; duplicated here so Notebook 04 is self-contained)

class LSTMOrbit(nn.Module):
    def __init__(self, n_features, hidden, num_layers, n_out, dropout):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        fc_in = hidden * 2
        self.head = nn.Sequential(
            nn.LayerNorm(fc_in), nn.Linear(fc_in, fc_in // 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(fc_in // 2, n_out))
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        ctx = torch.cat([h[-2], h[-1]], dim=-1)
        return self.head(ctx)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:d_model // 2])
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class TransformerOrbit(nn.Module):
    def __init__(self, n_features, d_model, nhead, num_encoder_layers, ff_dim, n_out, dropout):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc    = PositionalEncoding(d_model, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, ff_dim, dropout=dropout,
                                               batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model // 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model // 2, n_out))
    def forward(self, x):
        x = self.pos_enc(self.input_proj(x))
        return self.head(self.encoder(x).mean(dim=1))

def load_model(cls, path, device):
    ckpt = torch.load(path, map_location=device)
    hp = ckpt['hparams']
    model = cls(**{k: v for k, v in hp.items()})
    model.load_state_dict(ckpt['model_state'])
    model.to(device).eval()
    return model

lstm_model = load_model(
    lambda **kw: LSTMOrbit(kw['n_features'], kw['hidden'], kw['lstm_layers'],
                           kw['n_out'], kw['dropout']),
    DATA_DIR / 'lstm_orbit.pt', DEVICE)

tf_model = load_model(
    lambda **kw: TransformerOrbit(kw['n_features'], kw['d_model'], kw['nhead'],
                                  kw['num_encoder_layers'], kw['ff_dim'],
                                  kw['n_out'], kw['dropout']),
    DATA_DIR / 'transformer_orbit.pt', DEVICE)

print('Models loaded successfully.')

## 2 — Run Inference & Build Results Table

Geodetic error is computed as the great-circle distance on the surface (lat/lon) plus altitude difference added in quadrature — giving a combined 3-D position error in km.

In [ ]:
def geodetic_error_km(pred, true):
    """
    pred, true: arrays of shape (..., 3) with columns [lat_deg, lon_deg, alt_km].
    Returns 3-D position error in km for each sample.
    """
    lat1 = np.radians(pred[..., 0]); lat2 = np.radians(true[..., 0])
    lon1 = np.radians(pred[..., 1]); lon2 = np.radians(true[..., 1])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    d_surface = 2 * (RE_KM + true[..., 2]) * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    d_alt = pred[..., 2] - true[..., 2]
    return np.sqrt(d_surface**2 + d_alt**2)

def predict_all(model, X, batch=512):
    model.eval()
    parts = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            Xb = torch.tensor(X[i:i+batch], dtype=torch.float32).to(DEVICE)
            parts.append(model(Xb).cpu().numpy())
    return np.concatenate(parts, axis=0).reshape(-1, len(HORIZONS), 3)

# --- SGP4 baseline: use last known position (dead-reckoning = last TLE propagation endpoint)
# The last step in the input window IS the SGP4-propagated position, so predicting
# "no change from last known" is the naive SGP4 baseline for short horizons.
# For a fairer comparison we copy last-step lat/lon/alt as the prediction for all horizons.
feat_idx_lat = FEATURE_COLS.index('lat_deg')
feat_idx_lon = FEATURE_COLS.index('lon_deg')
feat_idx_alt = FEATURE_COLS.index('alt_km')
feat_idx_tle = FEATURE_COLS.index('tle_age_days')

# Un-scale the last input step to get the last known position
X_test_orig_shape = X_test.shape
X_test_2d = X_test.reshape(-1, X_test.shape[-1])
X_unscaled = scaler_X.inverse_transform(X_test_2d).reshape(X_test_orig_shape)

last_step = X_unscaled[:, -1, :]  # (N, F) — last timestep of each window
sgp4_pred = np.stack([last_step[:, [feat_idx_lat, feat_idx_lon, feat_idx_alt]]] * len(HORIZONS), axis=1)

# --- Model predictions ---
lstm_pred = predict_all(lstm_model, X_test)
tf_pred   = predict_all(tf_model,   X_test)

print(f'sgp4_pred : {sgp4_pred.shape}')
print(f'lstm_pred : {lstm_pred.shape}')
print(f'tf_pred   : {tf_pred.shape}')
print(f'y_test    : {y_test.shape}')

## 3 — Metric Table (MAE / RMSE by horizon)

In [ ]:
rows = []
for hi, h in enumerate(HORIZONS):
    for model_name, preds in [('SGP4 baseline', sgp4_pred),
                               ('LSTM',         lstm_pred),
                               ('Transformer',  tf_pred)]:
        err = geodetic_error_km(preds[:, hi, :], y_test[:, hi, :])
        rows.append({'Horizon (min)': h, 'Model': model_name,
                     'MAE (km)': round(err.mean(), 3),
                     'RMSE (km)': round(np.sqrt((err**2).mean()), 3)})

metrics_df = pd.DataFrame(rows)
pivot = metrics_df.pivot_table(index='Model', columns='Horizon (min)',
                                values=['MAE (km)', 'RMSE (km)'], aggfunc='first')
print(pivot.to_string())
pivot

## 4 — Error vs TLE Age

Does the ML model outperform SGP4 when the TLE is stale?  
Split test samples by TLE age quartile and compare MAE at T+60 min.

In [ ]:
H60_IDX = HORIZONS.index(60)

tle_age = last_step[:, feat_idx_tle]  # TLE age at end of each input window
age_bins = [0, 1, 3, 7, 100]
age_labels = ['<1 day', '1-3 days', '3-7 days', '>7 days']
age_group = pd.cut(tle_age, bins=age_bins, labels=age_labels)

age_rows = []
for grp in age_labels:
    mask = (age_group == grp)
    if mask.sum() == 0:
        continue
    for model_name, preds in [('SGP4', sgp4_pred), ('LSTM', lstm_pred), ('Transformer', tf_pred)]:
        err = geodetic_error_km(preds[mask, H60_IDX, :], y_test[mask, H60_IDX, :])
        age_rows.append({'TLE Age': grp, 'Model': model_name,
                         'MAE (km)': err.mean(), 'n': int(mask.sum())})

age_df = pd.DataFrame(age_rows)
fig = px.bar(age_df, x='TLE Age', y='MAE (km)', color='Model', barmode='group',
             title='MAE at T+60 min by TLE Age',
             labels={'MAE (km)': 'MAE (km)', 'TLE Age': 'TLE Age at Window End'},
             category_orders={'TLE Age': age_labels},
             template='plotly_white')
fig.show()
print(age_df.pivot_table(index='TLE Age', columns='Model', values='MAE (km)').round(3).to_string())

## 5 — Example Prediction Trajectories

Visualise 3 random test windows: ground truth vs LSTM vs Transformer (altitude over horizon).

In [ ]:
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X_test), size=3, replace=False)
ALT_IDX = TARGET_COLS.index('alt_km')

for i, idx in enumerate(sample_idx):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=HORIZONS, y=y_test[idx, :, ALT_IDX],
                             mode='lines+markers', name='Ground truth', line=dict(color='black')))
    fig.add_trace(go.Scatter(x=HORIZONS, y=lstm_pred[idx, :, ALT_IDX],
                             mode='lines+markers', name='LSTM', line=dict(color='royalblue', dash='dash')))
    fig.add_trace(go.Scatter(x=HORIZONS, y=tf_pred[idx, :, ALT_IDX],
                             mode='lines+markers', name='Transformer', line=dict(color='firebrick', dash='dot')))
    fig.add_trace(go.Scatter(x=HORIZONS, y=sgp4_pred[idx, :, ALT_IDX],
                             mode='lines+markers', name='SGP4 baseline', line=dict(color='grey', dash='longdash')))
    fig.update_layout(title=f'Sample {i+1}  (TLE age ≈ {tle_age[idx]:.1f} days)',
                      xaxis_title='Horizon (min)', yaxis_title='Altitude (km)',
                      template='plotly_white')
    fig.show()